# Subreddit groups v6 (v4 sample, pre-accusation subreddits)

Builds on `subreddit_groups_v5_banned_private_as_flags.ipynb` with the 2026-10-07 decisions:
- **new `test` group**: (A) Reddit-internal test subreddits — every name starting `lss` (all 60 have gibberish or
  "Mobile Testing Community" bios) plus `RedditMobTest`; (B) general test/sandbox subreddits listed below. `test` takes
  priority over every group except `profile` (8 of them were `adult` by NSFW posts);
- **self-promotion → `sales`**: subreddits for promoting one's own channel/stream/music/content (and two brand-promotion
  subreddits) that v5 had in `other`;
- YouTube/Twitch sub4sub and follow4follow exchanges stay in `karma_farming` (no change).

Groups: profile, test, adult, crypto, politics, karma_farming, general, news, sales, location, other; flags
`later_banned` and `private` unchanged. Outputs (never overwritten): `subreddit_groups_v6.parquet` and its review spreadsheet.

In [ ]:
# Libraries, folders, and the hand-picked lists: test subreddits, and self-promotion subreddits that move to sales.
import pandas as pd
import numpy as np
import os
import re
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

SAMPLE_DIR = 'sample_1000_v4_output'
OUT_DIR = 'subreddit_review_output'
TEST_GENERAL = ['RedditMobTest', 'test', 'testingground4bots', 'pub00', 'automodspamtest', 'bottest', 'redditgamingtest',
                'Smoketestingrulesplay', 'test23456', 'testing00002323', 'goldenboi']
SELF_PROMOTION = ['Twitch_Startup', 'SmallYTChannel', 'GetMoreViewsYT', 'TwitchPromotion', 'YoutubeGamingVidPromo',
                  'Promote_Your_Channel', 'promotereddit', 'mymusic', 'SmallYoutubers', 'YouTubersCommunity', 'funnyrobots211',
                  'starting_youtubers', 'SmallYTChannels', 'StreamWatchParty', 'MixersUnite', 'Smallyoutubechannels',
                  'shareyourmusic', 'YouTubePromoter', 'HDViewer', 'vpn_VeePN']

# Write a file only if it does not exist yet (never overwrite); .xlsx goes to Excel, anything else to parquet.
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    if name.endswith('.xlsx'):
        df.to_excel(path, index=False, freeze_panes=(1, 1))
    else:
        df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

In [ ]:
# Start from v5 and keep the v5 group alongside.
g = pd.read_parquet(os.path.join(OUT_DIR, 'subreddit_groups_v5.parquet'))
assert set(TEST_GENERAL + SELF_PROMOTION) <= set(g['subreddit']), set(TEST_GENERAL + SELF_PROMOTION) - set(g['subreddit'])
g['group_v5'] = g['group']
# Test group: names starting with lss (Reddit internal) plus the hand-picked list.
is_test = g['subreddit'].str.lower().str.startswith('lss') | g['subreddit'].isin(TEST_GENERAL)
is_promo = g['subreddit'].isin(SELF_PROMOTION)
assert not (is_test & g['group'].eq('profile')).any()
# Apply test first; self-promotion subreddits become sales unless they are test subreddits.
g.loc[is_test, ['group', 'group_source']] = ['test', 'rule_test_subreddit']
g.loc[is_promo & ~is_test, ['group', 'group_source']] = ['sales', 'rule_self_promotion']
# Which v5 groups the moved subreddits came from.
print(pd.crosstab(g['group_v5'], g['group'])[['test', 'sales']].loc[lambda d: d.sum(axis=1) > 0].to_string())

In [ ]:
# Every post and comment the sample made before its first (edit-adjusted) accusation, tagged with its subreddit's group.
accounts = pd.read_parquet(os.path.join(SAMPLE_DIR, 'sample_accounts.parquet')).set_index('username_key')
acts = []
for f in ['sample_posts.parquet', 'sample_comments.parquet']:
    d = pd.read_parquet(os.path.join(SAMPLE_DIR, f), columns=['outer_key', 'created_utc', 'subreddit'])
    acts.append(d[d['created_utc'] < d['outer_key'].map(accounts['first_mention_adj_utc'])])
acts = pd.concat(acts)
acts['group'] = acts['subreddit'].map(g.set_index('subreddit')['group'])
assert acts['group'].notna().all()
GROUPS = ['profile', 'test', 'adult', 'crypto', 'politics', 'karma_farming', 'general', 'news', 'sales', 'location', 'other']
# Per group: subreddits, actions and accounts, and the share of all pre-accusation actions.
summary = pd.DataFrame({'subreddits': g['group'].value_counts(), 'actions': acts['group'].value_counts(),
                        'accounts': acts.groupby('group')['outer_key'].nunique()}).reindex(GROUPS)
summary['pct_actions'] = (100 * summary['actions'] / summary['actions'].sum()).round(1)
summary

In [ ]:
# Save the groups, and a review sheet sorted by group and use.
save_new(g, 'subreddit_groups_v6.parquet')
review = g.sort_values(['group', 'actions'], ascending=[True, False])[
    ['subreddit', 'group', 'group_v5', 'group_source', 'confidence', 'reason', 'later_banned', 'private',
     'archived_nsfw_share', 'actions', 'accounts', 'title', 'public_description']]
save_new(review, 'subreddit_groups_v6_review.xlsx')